Step 1: Data Preprocessing

In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler,RobustScaler,MaxAbsScaler,StandardScaler,PowerTransformer
from torch.utils.data import DataLoader, TensorDataset

# Set a fixed random seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

def create_9x9_matrix(data_row):
    numeric_row = pd.to_numeric(data_row, errors='coerce').fillna(0)
    num_elements_required = 81
    if len(numeric_row) < num_elements_required:
        numeric_row = np.pad(numeric_row, (0, num_elements_required - len(numeric_row)), 'constant')
    matrix = np.array(numeric_row).reshape(9, 9)
    return matrix

def process_dataset(dataset):
    matrices = []
    for _, row in dataset.iterrows():
        matrix = create_9x9_matrix(row)
        matrices.append(matrix)
    return matrices


# Load dataset
dataset_path = 'random_samples.csv'  # Update this path
df = pd.read_csv(dataset_path)


# Convert columns to numeric, coercing errors will turn 'Infinity' and '-Infinity' into NaN
df['FlowBytes/s'] = pd.to_numeric(df['FlowBytes/s'], errors='coerce')
df['FlowPackets/s'] = pd.to_numeric(df['FlowPackets/s'], errors='coerce')

# Replace infinite values with NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Now, replace NaN in 'Flow Bytes/s' and 'Flow Packets/s' with their respective max value + 1
df['FlowBytes/s'].fillna(df['FlowBytes/s'].max() + 1, inplace=True)
df['FlowPackets/s'].fillna(df['FlowPackets/s'].max() + 1, inplace=True)

# Preprocessing
features = df.iloc[:, :-1].values
labels = df.iloc[:, -1].values

# Normalize the features
scaler = MinMaxScaler()
features = scaler.fit_transform(features)

# Process each row to create 9x9 matrices
matrices = process_dataset(pd.DataFrame(features))

# Convert to a 3D numpy array
X = np.array(matrices)

# Encode the labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)

# Split data
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED)

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=SEED)
# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

Step 2: Building the PDAE Model

In [2]:
import torch
import torch.nn as nn

class DeepAutoencoder(nn.Module):
    def __init__(self, num_input_features, num_classes, dropout_rate=0.4):
        super(DeepAutoencoder, self).__init__()
        self.num_input_features = num_input_features

        # Standard Encoder
        self.encoder_standard = nn.Sequential(
            nn.Linear(num_input_features, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 32)
        )

        # Adjust input size for Conv1d with dilation=3
        conv1d_output_size_dilated3 = (num_input_features - 3 * (3 - 1) - 1) + 1  # Adjusted for dilation=3
        conv1d_output_size_dilated2 = (num_input_features - 2 * (3 - 1) - 1) + 1  # Adjusted for dilation=2

        # Dilated Encoder with dilation factor of 3
        self.encoder_dilated3 = nn.Sequential(
            nn.Linear(num_input_features, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Conv1d(in_channels=1, out_channels=1, kernel_size=3, stride=1, dilation=3),  # Adjusted dilation
            nn.Flatten(),
            nn.Linear(58, 32)
        )

        # Dilated Encoder with dilation factor of 2
        self.encoder_dilated2 = nn.Sequential(
            nn.Linear(num_input_features, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Conv1d(in_channels=1, out_channels=1, kernel_size=3, stride=1, dilation=2),  # Adjusted dilation
            nn.Flatten(),
            nn.Linear(60, 32)
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(96, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, num_input_features)
        )

        # Dilated Decoder with adjusted input and output sizes for dilation=3
        self.decoder_dilated3 = nn.Sequential(
            nn.Linear(32, conv1d_output_size_dilated3),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.ConvTranspose1d(1, 1, kernel_size=3, stride=1, dilation=3),  # Adjusted dilation
            nn.Flatten(),
            nn.Linear(conv1d_output_size_dilated3, num_input_features)
        )

        # Dilated Decoder with adjusted input and output sizes for dilation=2
        self.decoder_dilated2 = nn.Sequential(
            nn.Linear(32, conv1d_output_size_dilated2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.ConvTranspose1d(1, 1, kernel_size=3, stride=1, dilation=2),  # Adjusted dilation
            nn.Flatten(),
            nn.Linear(conv1d_output_size_dilated2, num_input_features)
        )

        # Classifier
        self.classifier = nn.Linear(96, num_classes)

    def forward(self, x):
        x = x.view(-1, self.num_input_features)
        encoded_standard = self.encoder_standard(x)
        encoded_dilated3 = self.encoder_dilated3(x.unsqueeze(1))
        encoded_dilated2 = self.encoder_dilated2(x.unsqueeze(1))
        encoded_combined = torch.cat((encoded_standard, encoded_dilated3, encoded_dilated2), dim=1)
        decoded = self.decoder(encoded_combined)
        classification = self.classifier(encoded_combined)
        return decoded, classification
        


# Example usage
num_classes = len(label_encoder.classes_)  # Update this based on your dataset
num_input_features = 81  # Update this based on your dataset
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeepAutoencoder(num_input_features, num_classes).to(device)


Step 3: Training the Model

In [3]:
import torch.optim as optim  # Import the optim module

LEARNING_RATE = 0.01  # Adjusted learning rate
BATCH_SIZE = 64      # Adjusted batch size

criterion_reconstruction = nn.L1Loss()
criterion_classification = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_data = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_data = TensorDataset(X_val_tensor, y_val_tensor)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)

num_epochs = 100

for epoch in range(num_epochs):
    model.train()
    running_loss_reconstruction = 0.0
    running_loss_classification = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        decoded, classification = model(inputs.view(-1, num_input_features))
        loss_reconstruction = criterion_reconstruction(decoded, inputs.view(-1, num_input_features))
        loss_classification = criterion_classification(classification, labels)
        loss = loss_reconstruction + loss_classification
        loss.backward()
        optimizer.step()

        running_loss_reconstruction += loss_reconstruction.item()
        running_loss_classification += loss_classification.item()

    print(f'Epoch [{epoch+1}/{num_epochs}], Reconstruction Loss: {running_loss_reconstruction / len(train_loader)}, Classification Loss: {running_loss_classification / len(train_loader)}')

    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            decoded, outputs = model(inputs.view(-1, num_input_features)) # فرض بر این است که مدل دو خروجی دارد
            _, predicted = torch.max(outputs, 1) # 'outputs' به جای 'outputs.data' استفاده شده است
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    validation_accuracy = correct / total
    print(f'Epoch {epoch+1}, Validation Accuracy: {validation_accuracy:.4f}')




Epoch [1/100], Reconstruction Loss: 0.03188642466972981, Classification Loss: 0.15367672232193014
Epoch 1, Validation Accuracy: 0.9654
Epoch [2/100], Reconstruction Loss: 0.03114796675810951, Classification Loss: 0.111678972898566
Epoch 2, Validation Accuracy: 0.9660
Epoch [3/100], Reconstruction Loss: 0.03103279080718208, Classification Loss: 0.10065019984231353
Epoch 3, Validation Accuracy: 0.9677
Epoch [4/100], Reconstruction Loss: 0.031037276089962946, Classification Loss: 0.09276722784129318
Epoch 4, Validation Accuracy: 0.9706
Epoch [5/100], Reconstruction Loss: 0.031233554622922044, Classification Loss: 0.09123817931310886
Epoch 5, Validation Accuracy: 0.9717
Epoch [6/100], Reconstruction Loss: 0.031103571493293545, Classification Loss: 0.08708781482170337
Epoch 6, Validation Accuracy: 0.9683
Epoch [7/100], Reconstruction Loss: 0.031086136524377365, Classification Loss: 0.0889413631516294
Epoch 7, Validation Accuracy: 0.9766
Epoch [8/100], Reconstruction Loss: 0.0311106211141331

Step 4: Evaluating the Model and Calculating Metrics

In [4]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


model.eval()
true_labels = []
predicted_labels = []

test_data = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.view(-1, num_input_features)
        inputs, labels = inputs.to(device), labels.to(device)
        decoded, classification = model(inputs)
        _, predicted = torch.max(classification, 1)
        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predicted.cpu().tolist())

accuracy = accuracy_score(true_labels, predicted_labels)
precision = precision_score(true_labels, predicted_labels, average='weighted', zero_division=1)
recall = recall_score(true_labels, predicted_labels, average='weighted', zero_division=1)
f1 = f1_score(true_labels, predicted_labels, average='weighted', zero_division=1)
conf_matrix = confusion_matrix(true_labels, predicted_labels)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Confusion Matrix:\n", conf_matrix)

Accuracy: 0.981323689685921
Precision: 0.9810131051586405
Recall: 0.981323689685921
F1 Score: 0.9798644242622527
Confusion Matrix:
 [[83822     0     7     0     1     2     1     0     0   230     0     0
      0]
 [   83     0     0     0     0     0     0     0     0     0     0     0
      0]
 [   58     0  5043     0     0     0     0     0     0     0     0     0
      0]
 [   32     0     0   357     0     2     0     0     0     0     0     0
      0]
 [  374     0     3     0  6426     0     0     0     0     0     0     0
      0]
 [    4     0     0     0     0   182    27     0     0     0     0     0
      0]
 [   20     0     0     0     0     0   202     4     0     0     0     0
      0]
 [    2     0     0     0     0     0     0   234     0     0     0     0
      0]
 [    1     0     0     0     0     0     0     0     0     0     0     0
      0]
 [  938     0     2     0     1     0     0     0     0  2655     0     0
      0]
 [   10     0     0     0     0     0 

In [2]:
import csv

# آرایه‌ی داده جدید
new_data = [
    [83822, 0, 7, 0, 1, 2, 1, 0, 0, 230, 0, 0, 0],
    [83, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [58, 0, 5043, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [32, 0, 0, 357, 0, 2, 0, 0, 0, 0, 0, 0, 0],
    [374, 0, 3, 0, 6426, 0, 0, 0, 0, 0, 0, 0, 0],
    [4, 0, 0, 0, 0, 182, 27, 0, 0, 0, 0, 0, 0],
    [20, 0, 0, 0, 0, 0, 202, 4, 0, 0, 0, 0, 0],
    [2, 0, 0, 0, 0, 0, 0, 234, 0, 0, 0, 0, 0],
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [938, 0, 2, 0, 1, 0, 0, 0, 0, 2655, 0, 0, 0],
    [10, 0, 0, 0, 0, 0, 0, 1, 0, 0, 124, 0, 0],
    [57, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [25, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
]

# نام فایل CSV
csv_file_new = "CICIDS2017_D3_D2.csv"

# نوشتن داده‌های جدید به فایل CSV
with open(csv_file_new, mode='w', newline='', encoding='utf-8-sig') as file:
    writer = csv.writer(file)
    writer.writerows(new_data)

print(f"فایل {csv_file_new} با موفقیت ایجاد شد.")


فایل CICIDS2017_D3_D2.csv با موفقیت ایجاد شد.
